# Indexes & Query Plans

## Quick notes
### What this covers
- What an index is, how SQLite uses it, and how to see that with `EXPLAIN QUERY PLAN`.
- Covering, composite, partial, expression and UNIQUE indexes.
- Why an index is sometimes ignored, what indexes cost on writes, and how ANALYZE helps the planner.
### Key points
- An index is a sorted copy of some columns plus a pointer to the row (a B-tree). Finding a value is a few steps, not a full read.
- `SCAN` = read every row. `SEARCH ... USING INDEX` = jump straight to matching rows.
- A **covering index** holds every column the query needs, so the table is never touched.
- Composite index `(a, b, c)` helps queries on `a`, on `a,b`, on `a,b,c` (leftmost prefix) but not on `b` alone.
- Put equality columns first, then the range or sort column.
- Every index makes INSERT/UPDATE/DELETE a bit slower and uses space.
### Gotchas (the part worth revising twice)
- A function on the column (`lower(email) = ...`, `substr(...)`) blocks a plain index. Use an expression index or rewrite the query.
- `LIKE '%abc'` (leading wildcard) cannot use an index. `LIKE 'abc%'` only can when the index is NOCASE (or use GLOB).
- `OR` across different columns often falls back to a scan unless each side has an index.
- Collation mismatch (index is BINARY, query says NOCASE) means no index.
- Low-selectivity columns (e.g. a yes/no flag) rarely benefit from an index.
- Do not guess: run `EXPLAIN QUERY PLAN` and read it. Do not measure speed in tutorials; clocks differ, plans do not.
### When it matters
- Queries that filter, join, or sort on a column of a large table, run often.
- Foreign key columns used in joins and in `ON DELETE` checks.
### One-line recap example
```sql
CREATE INDEX idx_orders_cust_date ON orders (customer_id, order_date);
EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 7 ORDER BY order_date;
```

## 01. Seed data: a 2000-row table
- A recursive CTE counts 1 to 2000 and the INSERT turns each number into an order.
- Everything is computed from the number with `%` (remainder), so the data is the same every run.
- Columns: customer (1 to 200), status (4 values), a fixed-style date, an amount, and an email-like text.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    status      TEXT NOT NULL,
    order_date  TEXT NOT NULL,
    amount      INTEGER NOT NULL,
    email       TEXT NOT NULL
);

WITH RECURSIVE n(i) AS (
    SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 2000
)
INSERT INTO orders
SELECT i,
       1 + (i * 37) % 200,
       CASE i % 4 WHEN 0 THEN 'new' WHEN 1 THEN 'paid' WHEN 2 THEN 'shipped' ELSE 'returned' END,
       '2024-' || printf('%02d', 1 + (i % 12)) || '-' || printf('%02d', 1 + (i * 7) % 28),
       10 + (i * 13) % 490,
       'User' || (i % 500) || '@Example.com'
FROM n;

SELECT COUNT(*) AS rows_loaded, MIN(order_id) AS first_id, MAX(order_id) AS last_id FROM orders;

rows_loaded | first_id | last_id
------------+----------+--------
2000        | 1        | 2000   
(1 row)



## 02. B-tree intuition
- Think of a phone book: sorted by name, so you open near the middle, then narrow down. About log2(N) steps: for 2000 rows, roughly 11 steps instead of 2000.
- An index is exactly that: a sorted structure (a B-tree) of the indexed values, each with the row's rowid.
- The table itself is also a B-tree, sorted by rowid. That is why `WHERE order_id = 5` is already a fast SEARCH with no extra index.
- The price: every change to indexed values must also update the sorted structure.

In [3]:
%%sql
EXPLAIN QUERY PLAN SELECT * FROM orders WHERE order_id = 1500;

id | parent | notused | detail                                           
---+--------+---------+--------------------------------------------------
2  | 0      | 0       | SEARCH orders USING INTEGER PRIMARY KEY (rowid=?)
(1 row)



## 03. Reading EXPLAIN QUERY PLAN
- It shows the plan without running the query. Columns: `id`, `parent` (tree nesting), `notused` (ignore), `detail` (the useful text).
- `SCAN orders`: every row is read.
- `SEARCH orders USING INDEX name (col=?)`: the index is used to jump to rows where col equals a value.
- `USING COVERING INDEX`: the index alone answers the query.
- `USE TEMP B-TREE FOR ORDER BY`: SQLite had to sort the rows itself (a cost).

In [4]:
%%sql
EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 42;

id | parent | notused | detail     
---+--------+---------+------------
2  | 0      | 0       | SCAN orders
(1 row)



## 04. CREATE INDEX: SCAN becomes SEARCH
- Same query, now with an index on the filtered column.
- The result rows are identical; only the plan (and the work done) changes.

In [5]:
%%sql
CREATE INDEX idx_orders_customer ON orders (customer_id);
EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 42;
SELECT order_id, status, amount FROM orders WHERE customer_id = 42 ORDER BY order_id;

id | parent | notused | detail                                                       
---+--------+---------+--------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING INDEX idx_orders_customer (customer_id=?)
(1 row)

order_id | status | amount
---------+--------+-------
93       | paid   | 239   
293      | paid   | 389   
493      | paid   | 49    
693      | paid   | 199   
893      | paid   | 349   
1093     | paid   | 499   
1293     | paid   | 159   
1493     | paid   | 309   
1693     | paid   | 459   
1893     | paid   | 119   
(10 rows)



## 05. Equality, ranges, and IN
- A B-tree handles `=`, `<`, `>`, `BETWEEN`, and `IN (...)` (one lookup per value).
- Ranges are a SEARCH too, using `>?` and `<?` bounds.

In [6]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id BETWEEN 10 AND 12;
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id IN (5, 6, 7);

id | parent | notused | detail                                                                                  
---+--------+---------+-----------------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id>? AND customer_id<?)
(1 row)

id | parent | notused | detail                                                                
---+--------+---------+-----------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id=?)
(1 row)



## 06. Covering indexes
- If the index holds every column the query reads, SQLite never opens the table: it reads only the index.
- `customer_id` index also stores the rowid, so `SELECT order_id ... WHERE customer_id = ?` is already covered.
- Selecting another column (`amount`) forces a trip to the table. Add it to the index to cover the query.
- Trade-off: wider index = more space and slower writes. Cover only your hottest queries.

In [7]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = 42;
EXPLAIN QUERY PLAN SELECT amount FROM orders WHERE customer_id = 42;
CREATE INDEX idx_orders_customer_amount ON orders (customer_id, amount);
EXPLAIN QUERY PLAN SELECT amount FROM orders WHERE customer_id = 42;

id | parent | notused | detail                                                                
---+--------+---------+-----------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id=?)
(1 row)

id | parent | notused | detail                                                       
---+--------+---------+--------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING INDEX idx_orders_customer (customer_id=?)
(1 row)

id | parent | notused | detail                                                                       
---+--------+---------+------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer_amount (customer_id=?)
(1 row)



## 07. Composite index and the leftmost-prefix rule
- An index on `(status, order_date)` is sorted by status first, then date inside each status.
- Usable for: `status = ?`, or `status = ? AND order_date ...` (a SEARCH). For `order_date` alone it is not usable for a lookup, like searching a phone book by first name only: the best SQLite can do is SCAN the whole index (smaller than the table, but still every entry).
- Rule of thumb: equality columns first, then the range/sort column.

In [8]:
%%sql
CREATE INDEX idx_orders_status_date ON orders (status, order_date);
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status = 'paid';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status = 'paid' AND order_date >= '2024-06-01';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE order_date = '2024-06-05';

id | parent | notused | detail                                                              
---+--------+---------+---------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_status_date (status=?)
(1 row)

id | parent | notused | detail                                                                               
---+--------+---------+--------------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_status_date (status=? AND order_date>?)
(1 row)

id | parent | notused | detail                                                 
---+--------+---------+--------------------------------------------------------
2  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_status_date
(1 row)



## 08. Column order matters
- Two indexes with the same columns in different order serve different queries.
- Bad vs good for "paid orders": an index starting with `status` can SEARCH. The same columns as `(order_date, status)` cannot, because `status` is not the leading column.
- Below: the same predicate against a copy of the table with the wrong-order index (bad) and the original (good).

In [9]:
%%sql
CREATE TABLE orders_date_first AS SELECT * FROM orders;
CREATE INDEX idx_df_date_status ON orders_date_first (order_date, status);
EXPLAIN QUERY PLAN SELECT order_id FROM orders_date_first WHERE status = 'paid';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status = 'paid';

id | parent | notused | detail                
---+--------+---------+-----------------------
2  | 0      | 0       | SCAN orders_date_first
(1 row)

id | parent | notused | detail                                                              
---+--------+---------+---------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_status_date (status=?)
(1 row)



## 09. UNIQUE indexes
- `CREATE UNIQUE INDEX` enforces "no duplicates" and speeds lookups. A UNIQUE constraint creates one behind the scenes.
- Creating it fails if duplicates already exist (our 2000 orders only have 500 distinct emails), so clean the data first. A `users` table with one row per email can have it.
- Many NULLs are still allowed (see 05-DDL-And-Integrity/01-Constraints-And-Keys).

In [10]:
%%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT email) AS distinct_emails FROM orders;

rows | distinct_emails
-----+----------------
2000 | 500            
(1 row)



In [11]:
%%sql expect_error
CREATE UNIQUE INDEX uq_orders_email ON orders (email);

ERROR: UNIQUE constraint failed: orders.email



In [12]:
%%sql
CREATE TABLE users (user_id INTEGER PRIMARY KEY, email TEXT NOT NULL);
INSERT INTO users (email) SELECT DISTINCT email FROM orders ORDER BY email;
CREATE UNIQUE INDEX uq_users_email ON users (email);
EXPLAIN QUERY PLAN SELECT user_id FROM users WHERE email = 'User7@Example.com';
SELECT user_id, email FROM users WHERE email = 'User7@Example.com' ORDER BY user_id;

id | parent | notused | detail                                                    
---+--------+---------+-----------------------------------------------------------
2  | 0      | 0       | SEARCH users USING COVERING INDEX uq_users_email (email=?)
(1 row)

user_id | email            
--------+------------------
478     | User7@Example.com
(1 row)



In [13]:
%%sql expect_error
INSERT INTO users (email) VALUES ('User7@Example.com');

ERROR: UNIQUE constraint failed: users.email



## 10. Partial indexes
- Add a `WHERE` to the index: only matching rows are stored.
- Smaller, faster to maintain, perfect for "hot" subsets (open orders, active users).
- The query must include the same condition (or one that implies it) for SQLite to use it.

In [14]:
%%sql
CREATE INDEX idx_orders_returned ON orders (customer_id) WHERE status = 'returned';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status = 'returned' AND customer_id = 9;
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE status = 'paid' AND customer_id = 9;

id | parent | notused | detail                                                                
---+--------+---------+-----------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_returned (customer_id=?)
(1 row)

id | parent | notused | detail                                                              
---+--------+---------+---------------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING INDEX idx_orders_customer_amount (customer_id=?)
(1 row)



## 11. Expression indexes
- Index the result of an expression, e.g. `lower(email)`.
- The query must use the exact same expression to match.
- Our emails have mixed case ('User7@Example.com'), so a case-insensitive lookup is a realistic need.

In [15]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE lower(email) = 'user7@example.com';
CREATE INDEX idx_orders_email_lower ON orders (lower(email));
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE lower(email) = 'user7@example.com';

id | parent | notused | detail     
---+--------+---------+------------
2  | 0      | 0       | SCAN orders
(1 row)

id | parent | notused | detail                                                     
---+--------+---------+------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING INDEX idx_orders_email_lower (<expr>=?)
(1 row)



## 12. The index cost on writes
- Each index is another B-tree. An INSERT writes the table row and one entry per index. UPDATE of an indexed column also moves entries.
- So more indexes = faster reads on those paths, slower and bigger writes.
- We count instead of timing: list the indexes the table must maintain.
- Habit: index for real queries you can name; drop indexes nobody uses.

In [16]:
%%sql
SELECT name, "unique", partial FROM pragma_index_list('orders') ORDER BY name;
SELECT COUNT(*) AS btrees_touched_per_insert FROM pragma_index_list('orders');

name                       | unique | partial
---------------------------+--------+--------
idx_orders_customer        | 0      | 0      
idx_orders_customer_amount | 0      | 0      
idx_orders_email_lower     | 0      | 0      
idx_orders_returned        | 0      | 1      
idx_orders_status_date     | 0      | 0      
(5 rows)

btrees_touched_per_insert
-------------------------
5                        
(1 row)



## 13. When indexes are NOT used: functions on the column
- `WHERE substr(order_date, 1, 7) = '2024-06'` hides the column inside a function. The index on `order_date` cannot be used for a lookup (at best SQLite scans the whole index).
- Bad: wrap the column. Good: rewrite as a range on the bare column.

In [17]:
%%sql
CREATE INDEX idx_orders_date ON orders (order_date);
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE substr(order_date, 1, 7) = '2024-06';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE order_date >= '2024-06-01' AND order_date < '2024-07-01';

id | parent | notused | detail                                          
---+--------+---------+-------------------------------------------------
2  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_date
(1 row)

id | parent | notused | detail                                                                            
---+--------+---------+-----------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_date (order_date>? AND order_date<?)
(1 row)



## 14. When indexes are NOT used: LIKE with a leading wildcard
- `LIKE '%000'` needs to look at every value: SCAN.
- `LIKE 'User1%'` could use an index, but LIKE ignores case by default while a plain index is case-sensitive (BINARY), so SQLite refuses.
- Fixes: `GLOB 'User1*'` (case-sensitive, index-friendly), or an index declared `COLLATE NOCASE` so LIKE can use it.

In [18]:
%%sql
CREATE INDEX idx_orders_email ON orders (email);
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE email LIKE '%100@Example.com';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE email LIKE 'User1%';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE email GLOB 'User1*';

id | parent | notused | detail                                           
---+--------+---------+--------------------------------------------------
2  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_email
(1 row)

id | parent | notused | detail                                           
---+--------+---------+--------------------------------------------------
2  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_email
(1 row)

id | parent | notused | detail                                                                   
---+--------+---------+--------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_email (email>? AND email<?)
(1 row)



In [19]:
%%sql
CREATE INDEX idx_orders_email_nocase ON orders (email COLLATE NOCASE);
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE email LIKE 'User1%';

id | parent | notused | detail                                                                          
---+--------+---------+---------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_email_nocase (email>? AND email<?)
(1 row)



## 15. When indexes are NOT used: types and collations
- SQLite applies column "affinity" to compared values, so comparing an INTEGER column to `'42'` (text) or a TEXT column to `42` still works and still uses the index.
- The real trap is collation: if the query compares with a different collation than the index, the index is skipped.
- Another trap: arithmetic on the column (`amount + 0 = 20`).

In [20]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = '42';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE order_date = 20240605;
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE order_date = '2024-06-05' COLLATE NOCASE;
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id + 0 = 42;

id | parent | notused | detail                                                                
---+--------+---------+-----------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id=?)
(1 row)

id | parent | notused | detail                                                           
---+--------+---------+------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_date (order_date=?)
(1 row)

id | parent | notused | detail                                          
---+--------+---------+-------------------------------------------------
2  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_date
(1 row)

id | parent | notused | detail                                              
---+--------+---------+-----------------------------------------------------
2  | 0      | 0       | SCAN orders USING COV

## 16. When indexes are NOT used: OR
- `WHERE a = ? OR b = ?` can only use indexes if BOTH columns are indexed (SQLite does a "multi-index OR").
- If one side has no index, the whole thing falls back to a SCAN.
- Alternative: rewrite as two SELECTs joined by `UNION`.

In [21]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = 5 OR order_date = '2024-06-05';
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE amount = 100 OR email = 'x@y.com';

id | parent | notused | detail                                                              
---+--------+---------+---------------------------------------------------------------------
4  | 0      | 0       | MULTI-INDEX OR                                                      
5  | 4      | 0       | INDEX 1                                                             
11 | 5      | 0       | SEARCH orders USING INDEX idx_orders_customer_amount (customer_id=?)
16 | 4      | 0       | INDEX 2                                                             
22 | 16     | 0       | SEARCH orders USING INDEX idx_orders_date (order_date=?)            
(5 rows)

id | parent | notused | detail     
---+--------+---------+------------
2  | 0      | 0       | SCAN orders
(1 row)



## 17. Low selectivity and tiny tables
- `status` has only 4 values: each matches about a quarter of the table. An index there rarely beats reading the table once.
- With statistics from ANALYZE (see section 20) the planner can judge this better.
- Index columns that narrow the result to a small fraction: ids, emails, dates in ranges.

In [22]:
%%sql
SELECT status, COUNT(*) AS n FROM orders GROUP BY status ORDER BY status;
SELECT COUNT(DISTINCT customer_id) AS customers, COUNT(DISTINCT email) AS emails FROM orders;

status   | n  
---------+----
new      | 500
paid     | 500
returned | 500
shipped  | 500
(4 rows)

customers | emails
----------+-------
200       | 500   
(1 row)



## 18. ORDER BY and an index
- A B-tree is already sorted, so an index can deliver rows in order with no sort step.
- Without a suitable index you see `USE TEMP B-TREE FOR ORDER BY`.
- Index `(customer_id, amount)`: filter on the first column, and rows inside are already ordered by the second. No sort.
- `ORDER BY amount` alone has no index starting with `amount`, so a sort is needed.
- `(customer_id, order_date)` does the same for date ordering.

In [23]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = 42 ORDER BY amount;
EXPLAIN QUERY PLAN SELECT order_id FROM orders ORDER BY amount;
CREATE INDEX idx_orders_cust_date ON orders (customer_id, order_date);
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = 42 ORDER BY order_date;

id | parent | notused | detail                                                                       
---+--------+---------+------------------------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer_amount (customer_id=?)
(1 row)

id | parent | notused | detail                                                     
---+--------+---------+------------------------------------------------------------
3  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_customer_amount
10 | 0      | 0       | USE TEMP B-TREE FOR ORDER BY                               
(2 rows)

id | parent | notused | detail                                                                 
---+--------+---------+------------------------------------------------------------------------
3  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_cust_date (customer_id=?)
(1 row)



## 19. DISTINCT and GROUP BY using an index
- Duplicates and groups sit next to each other in an index, so SQLite can count or de-duplicate while walking it.
- Without an index it must build a temporary B-tree to group.
- Check for `USE TEMP B-TREE FOR DISTINCT` / `FOR GROUP BY`.

In [24]:
%%sql
EXPLAIN QUERY PLAN SELECT DISTINCT customer_id FROM orders;
EXPLAIN QUERY PLAN SELECT DISTINCT amount FROM orders;
EXPLAIN QUERY PLAN SELECT customer_id, COUNT(*) FROM orders GROUP BY customer_id;

id | parent | notused | detail                                              
---+--------+---------+-----------------------------------------------------
3  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_customer
(1 row)

id | parent | notused | detail                                                     
---+--------+---------+------------------------------------------------------------
3  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_customer_amount
11 | 0      | 0       | USE TEMP B-TREE FOR DISTINCT                               
(2 rows)

id | parent | notused | detail                                              
---+--------+---------+-----------------------------------------------------
6  | 0      | 0       | SCAN orders USING COVERING INDEX idx_orders_customer
(1 row)



## 20. ANALYZE and sqlite_stat1
- `ANALYZE` samples the tables and stores row counts per index in `sqlite_stat1`.
- The planner uses it to choose between indexes. Without it, it guesses.
- The `stat` text reads: total rows, then average rows per distinct value of the first column, first two columns, etc.
- Re-run `ANALYZE` after large data changes. `PRAGMA optimize` is the lightweight version for apps to call before closing.

In [25]:
%%sql
ANALYZE;
SELECT tbl, idx, stat FROM sqlite_stat1 WHERE tbl = 'orders' ORDER BY idx;

tbl    | idx                        | stat        
-------+----------------------------+-------------
orders | idx_orders_cust_date       | 2000 10 4   
orders | idx_orders_customer        | 2000 10     
orders | idx_orders_customer_amount | 2000 10 1   
orders | idx_orders_date            | 2000 167    
orders | idx_orders_email           | 2000 4      
orders | idx_orders_email_lower     | 2000 4      
orders | idx_orders_email_nocase    | 2000 4      
orders | idx_orders_returned        | 500 10      
orders | idx_orders_status_date     | 2000 500 167
(9 rows)



## 21. Planner hints: INDEXED BY and NOT INDEXED
- `FROM orders INDEXED BY idx_name` forces that index. If it cannot be used, the query FAILS (it does not fall back).
- `FROM orders NOT INDEXED` forbids all indexes (useful to compare plans).
- The error below appears because the partial index only holds 'returned' rows, so it cannot answer a query about 'paid' rows.
- Use for experiments, not as a permanent fix. Prefer fixing the query or running ANALYZE.

In [26]:
%%sql
EXPLAIN QUERY PLAN SELECT order_id FROM orders INDEXED BY idx_orders_customer WHERE customer_id = 42;
EXPLAIN QUERY PLAN SELECT order_id FROM orders NOT INDEXED WHERE customer_id = 42;

id | parent | notused | detail                                                                
---+--------+---------+-----------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer (customer_id=?)
(1 row)

id | parent | notused | detail     
---+--------+---------+------------
2  | 0      | 0       | SCAN orders
(1 row)



In [27]:
%%sql expect_error
SELECT order_id FROM orders INDEXED BY idx_orders_returned WHERE status = 'paid';

ERROR: no query solution



## 22. Cleaning up: drop unused indexes
- `DROP INDEX name` removes one. Tables and data are unaffected.
- `PRAGMA index_info(name)` lists the columns an index covers; useful for spotting duplicates and prefixes.
- Here `idx_orders_customer` is a prefix of `idx_orders_customer_amount` and of `idx_orders_cust_date`, so it is redundant. The wider index serves the same lookups.

In [28]:
%%sql
SELECT seqno, name FROM pragma_index_info('idx_orders_customer_amount') ORDER BY seqno;
DROP INDEX idx_orders_customer;
EXPLAIN QUERY PLAN SELECT order_id FROM orders WHERE customer_id = 42;

seqno | name       
------+------------
0     | customer_id
1     | amount     
(2 rows)

id | parent | notused | detail                                                                       
---+--------+---------+------------------------------------------------------------------------------
2  | 0      | 0       | SEARCH orders USING COVERING INDEX idx_orders_customer_amount (customer_id=?)
(1 row)



## 23. Checklist
- Name the slow query, then run `EXPLAIN QUERY PLAN` on it.
- See SCAN on a big table with a selective filter? Index the filter column(s).
- Equality columns first, then range/sort. Add covering columns only for hot queries.
- Check functions, leading wildcards, collation, OR.
- Foreign key columns used in joins deserve an index (see 02-Querying/01-Joins and 05-DDL-And-Integrity/01-Constraints-And-Keys).
- Re-check after `ANALYZE`; drop indexes you do not need.

In [29]:
%%sql
SELECT name FROM pragma_index_list('orders') ORDER BY name;

name                      
--------------------------
idx_orders_cust_date      
idx_orders_customer_amount
idx_orders_date           
idx_orders_email          
idx_orders_email_lower    
idx_orders_email_nocase   
idx_orders_returned       
idx_orders_status_date    
(8 rows)

